# Per-Pollutant Daily AQI Targets

The main dataset's `daily_aqi` is a **max across five pollutants**, and the models struggle
with that. On ozone-dominated test days mean absolute error is 16.3 with a +5.5 bias; on
PM2.5 days it is 10.8 with a −2.8 bias. Two different physical processes, opposite biases,
one function splitting the difference.

Ozone is photochemical — it needs heat and sunlight. PM2.5 is mechanical — inversions,
smoke, winter wood burning. A single model has to learn both *and* learn the discontinuous
switch between whichever is higher on a given day.

This notebook rebuilds the same AQI series but keeps the pollutants separate, so a model can
be fitted per pollutant and the daily maximum taken afterwards from the predictions.

Output: `data/processed/la_daily_aqi_by_pollutant_2016_2025.csv`

## Setup

Reads the same cached AQS API pulls the main pipeline uses, and applies the identical
filtering so the per-pollutant series are consistent with `daily_aqi`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

API_DIR   = ROOT / 'data' / 'interim' / 'api_pulls'
PROCESSED = ROOT / 'data' / 'processed'
OUT_CSV   = PROCESSED / 'la_daily_aqi_by_pollutant_2016_2025.csv'

assert API_DIR.exists(), (
    f'{API_DIR} is missing — it is gitignored. '
    'Run notebooks/aqi_pipeline.ipynb first to rebuild the API pulls.'
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

POL_LABEL = {'ozone': 'Ozone', 'pm25': 'PM2.5', 'co': 'CO', 'so2': 'SO2', 'no2': 'NO2'}

frames = []
for f in sorted(API_DIR.glob('*.csv')):
    df = pd.read_csv(f, low_memory=False)
    # pm25_88502_YYYY and pm25_YYYY both map to PM2.5 — the 88502 fix from aqi_pipeline
    df['pollutant_type'] = 'PM2.5' if f.name.startswith('pm25_88502_') \
                           else POL_LABEL[f.name.split('_')[0]]
    frames.append(df)

raw = pd.concat(frames, ignore_index=True)
print(f'{len(frames)} API files, {len(raw):,} raw rows')

60 API files, 700,463 raw rows


## Applying the same filters as the main pipeline

Identical to `aqi_pipeline.ipynb`: drop rows with no AQI, require at least 75% observation
completeness, then dedupe to one value per (site, poc, date, pollutant) by taking the max
across the multiple NAAQS standards that share the same underlying reading.

In [2]:
raw = raw.dropna(subset=['aqi'])
before = len(raw)
raw = raw[raw['observation_percent'] >= 75]
print(f'after aqi-not-null + observation_percent >= 75: {len(raw):,} (dropped {before-len(raw):,})')

assert (raw['state']  == 'California').all()
assert (raw['county'] == 'Los Angeles').all()

raw['date']    = pd.to_datetime(raw['date_local'], errors='coerce')
raw['site_id'] = (raw['state_code'].astype(str).str.zfill(2)
                + raw['county_code'].astype(str).str.zfill(3)
                + raw['site_number'].astype(str).str.zfill(4))

dedup = (raw.groupby(['date', 'site_id', 'poc', 'pollutant_type'], as_index=False)
            .agg(aqi=('aqi', 'max')))
print(f'after dedup to (site, poc, date, pollutant): {len(dedup):,}')
print()
print(dedup.groupby('pollutant_type')['aqi'].describe()[['count', 'mean', 'max']].round(1).to_string())

after aqi-not-null + observation_percent >= 75: 549,394 (dropped 8,280)


after dedup to (site, poc, date, pollutant): 194,341

                  count  mean    max
pollutant_type                      
CO              39422.0   5.5   62.0
NO2             51503.0  23.7  104.0
Ozone           45138.0  50.0  235.0
PM2.5           49494.0  50.8  250.0
SO2              8784.0   1.0   43.0


## One column per pollutant

For each day and pollutant, the city-wide value is the max across monitors — the same
convention `daily_aqi` uses, just not collapsed across pollutants.

In [3]:
wide = (dedup.groupby(['date', 'pollutant_type'])['aqi'].max()
             .unstack('pollutant_type'))
wide.columns = [f'aqi_{c.lower().replace(".", "").replace("2.5", "25")}' for c in wide.columns]
wide = wide.rename(columns={'aqi_pm25': 'aqi_pm25', 'aqi_ozone': 'aqi_ozone'})
wide = wide.reset_index()

print(f'{len(wide)} days, columns: {list(wide.columns)}')
print()
print('missing values per pollutant:')
print(wide.isna().sum().to_string())

3653 days, columns: ['date', 'aqi_co', 'aqi_no2', 'aqi_ozone', 'aqi_pm25', 'aqi_so2']

missing values per pollutant:
date         0
aqi_co       0
aqi_no2      0
aqi_ozone    0
aqi_pm25     0
aqi_so2      0


All five pollutants are complete — no missing days for any of them, which is better than
expected given CO and SO2 have far fewer monitors. Ozone and PM2.5 are the two that matter
here; as the main pipeline established, CO and SO2 never drive the daily maximum in LA.

## Checking this reconstructs the headline series

The row-wise max across these columns should reproduce `daily_aqi` exactly. If it does not,
the filtering has drifted from the main pipeline somewhere.

In [4]:
pol_cols = [c for c in wide.columns if c.startswith('aqi_')]
wide['daily_aqi_reconstructed'] = wide[pol_cols].max(axis=1)

official = pd.read_csv(PROCESSED / 'la_daily_aqi_5pollutants_v2_2016_2025.csv',
                       parse_dates=['date'])[['date', 'daily_aqi', 'dominant_pollutant']]
chk = wide.merge(official, on='date', how='inner')

mismatch = chk[chk['daily_aqi_reconstructed'] != chk['daily_aqi']]
print(f'days compared: {len(chk)}')
print(f'mismatches vs the official daily_aqi: {len(mismatch)}')
if len(mismatch):
    print(mismatch[['date', 'daily_aqi_reconstructed', 'daily_aqi']].head(10).to_string(index=False))

days compared: 3653
mismatches vs the official daily_aqi: 0


Zero mismatches, so the per-pollutant decomposition is exactly consistent with the headline
series. Any model built on these columns is predicting the same quantity, just factored.

## What the two main pollutants look like

Worth seeing before modeling: how often each one wins, and how differently they behave
through the year.

In [5]:
print('How often each pollutant sets the daily max:')
print(chk['dominant_pollutant'].value_counts().to_string())
print()
print('Distribution by pollutant:')
print(wide[['aqi_ozone', 'aqi_pm25']].describe(percentiles=[.5, .9, .99]).round(1).to_string())
print()
seasonal = (wide.assign(month=wide.date.dt.month)
                .groupby('month')[['aqi_ozone', 'aqi_pm25']].mean().round(1))
print('Monthly means — opposite seasonal cycles:')
print(seasonal.to_string())

How often each pollutant sets the daily max:
dominant_pollutant
PM2.5    1856
Ozone    1663
NO2       134

Distribution by pollutant:
       aqi_ozone  aqi_pm25
count     3653.0    3653.0
mean        75.0      67.6
std         44.2      19.0
min         23.0      23.0
50%         51.0      64.0
90%        146.2      85.0
99%        201.0     152.5
max        235.0     250.0

Monthly means — opposite seasonal cycles:
       aqi_ozone  aqi_pm25
month                     
1           37.2      72.6
2           44.0      62.7
3           50.0      56.8
4           70.9      60.6
5           77.9      62.1
6          110.2      66.2
7          127.6      72.8
8          124.3      68.7
9           98.5      69.3
10          76.6      68.2
11          45.3      71.6
12          35.7      79.5


This is the argument for splitting, in one table. **Ozone peaks in summer and PM2.5 peaks in
winter** — the two pollutants have nearly opposite seasonal cycles. A single model predicting
their max has to represent a summer regime driven by heat and a winter regime driven by
trapping, plus the changeover between them, all with one set of parameters.

## Saving

In [6]:
out = wide[['date'] + pol_cols].copy()
out.to_csv(OUT_CSV, index=False)
print(f'Saved {OUT_CSV.relative_to(ROOT)}  ({len(out)} rows, {out.shape[1]} columns)')
out.head()

Saved data/processed/la_daily_aqi_by_pollutant_2016_2025.csv  (3653 rows, 6 columns)


,date,aqi_co,aqi_no2,aqi_ozone,aqi_pm25,aqi_so2
0,2016-01-01,9.0,42.0,40.0,101.0,6.0
1,2016-01-02,24.0,42.0,37.0,83.0,4.0
2,2016-01-03,34.0,41.0,40.0,100.0,3.0
3,2016-01-04,13.0,48.0,38.0,68.0,1.0
4,2016-01-05,13.0,40.0,40.0,58.0,3.0
